# Training trajectory

Reproduces **Fig. 3a**, **Fig. 3b** and **Figs. S5, S9**.

How do a network's structure, its dynamics, and its behaviour come to agree? Tracking
all three across training shows they do not arrive together: geometry is laid down
first, correspondence with brain activity follows, and only then is the task mastered —
at which point the network partly relinquishes the geometry it started from.

**Requires:** the trajectory results pickle, written by
`scripts/biornn_results_dynamics_trajectory.py`, plus the HCP fMRI files for the
random-subspace null. Fig. 3b additionally re-evaluates the trained bioRNNs.

Analysis code lives in [`src/trajectory.py`](../src/trajectory.py),
[`src/dynamics.py`](../src/dynamics.py) and [`src/null_utils.py`](../src/null_utils.py).

> Variance explained is reported as **z against a random-subspace null** throughout,
> the same standardization used in the dynamics notebook, so values here are directly
> comparable with Figs. 2c and 2d.

In [ ]:
import os
import warnings

import gymnasium
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib import cm
from matplotlib.patches import Ellipse
from tqdm.notebook import tqdm

import src.dynamics as dyn
import src.null_utils as nu
import src.trajectory as traj
import src.utils as utils
from src.config import ensure_dir, get_paths
from src.fmri_io import load_fmri_data

# gymnasium routes deprecation notices through its own logger, which the
# warnings filter does not reach.
warnings.filterwarnings('ignore')
gymnasium.logger.min_level = gymnasium.logger.ERROR

MODEL_PARAMS = 'model_params_202606d'
MODALITY = 'task'         # empirical modality the trajectories are scored against
N_PC = 5
N_FMRI_SUBJ = 100
N_NULL_DRAWS = 1000
SAVE_FIGURES = True

paths = get_paths(MODEL_PARAMS, require='all')
figdir = ensure_dir(paths.figure_dir)

utils.set_font_size(11)
plt.rcParams['svg.fonttype'] = 'none'
sns.set_style('white')
COLORS = utils.get_my_colors(cat_trio=True, as_list=True)


def save(fig, name):
    if SAVE_FIGURES:
        fig.savefig(os.path.join(figdir, name), dpi=300,
                    bbox_inches='tight', pad_inches=0.01)


print(f'models : {paths.model_dir}')
print(f'figures: {figdir}')

## Inputs

The trajectory pickle holds, for every run of every class, a record at each of 100
training checkpoints. The null is rebuilt here from the same subjects the pickled
variance-explained values were computed against.

In [ ]:
trajectory = traj.load_trajectory(MODEL_PARAMS)
results = traj.class_results(trajectory)

fmri_task, fmri_rest, rest_nsteps, subjects = load_fmri_data(
    paths.data_dir, paths.fmri_dir, N_FMRI_SUBJ, hidden_size=100)

# The null must be calibrated on the cohort the stored VE values came from.
stored = [str(s) for s in trajectory['fmri_subjects']]
assert stored == [str(s) for s in subjects], \
    'subject set differs from the one used to compute the trajectory'

null = nu.random_subspace_null(
    {'task': fmri_task, 'rest': fmri_rest},
    n_pc=N_PC, n_draws=N_NULL_DRAWS, paired=False, seed=0)

labels = [r['kernel_label'].strip() for r in results]
print(f"\nclasses: {labels}")
print(f"checkpoints: {len(results[0]['sampled_epochs'])} "
      f"(epoch {results[0]['sampled_epochs'][0]}–{results[0]['sampled_epochs'][-1]})")

In [ ]:
# Reshape each class into (n_runs, n_epochs) arrays and standardize the VE.
series = {}
for result, label in zip(results, labels):
    arrays = traj.epoch_arrays(result, fields=('accuracy',),
                               modality=MODALITY, wk_metric='cosine')
    arrays['z'] = traj.variance_explained_z(arrays, null, MODALITY)
    series[label] = arrays

for label in labels:
    a = series[label]
    z_mean, _ = traj.mean_ci(a['z'])
    acc_mean, _ = traj.mean_ci(a['accuracy'])
    print(f'{label:<16} z {z_mean[0]:+.2f} → {z_mean[-1]:+.2f}   '
          f'accuracy {acc_mean[0]*100:.0f}% → {acc_mean[-1]*100:.0f}%')

## Fig. S9 — when each class predicts brain activity

The three classes follow different courses. Vanilla RNNs never rise above chance.
Masked RNNs show an early rise that fades as they master the task. Only bioRNNs
become and remain predictive.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))

for label, color in zip(labels, COLORS):
    a = series[label]
    mean, ci = traj.mean_ci(a['z'])
    ax.plot(a['epochs'], mean, color=color, lw=2, label=label)
    ax.fill_between(a['epochs'], mean - ci, mean + ci, color=color, alpha=0.15, lw=0)

ax.axhspan(-1.96, 1.96, color='0.85', zorder=0)   # chance band
ax.axhline(0, color='0.5', lw=1, zorder=1)
ax.set_xlabel('Training epoch')
ax.set_ylabel('fMRI variance explained\n($z$ vs random subspaces)')
ax.legend(frameon=False, loc='upper left')
sns.despine(fig=fig, right=True, top=True)
save(fig, 'figS9_fmri_prediction_timeline.svg')
plt.show()

## Fig. 3a — the hysteresis loop

Plotting fMRI prediction against weight–kernel similarity, coloured by task accuracy,
traces a loop rather than a line. The same structural similarity corresponds to very
different levels of brain-activity prediction depending on *when* in training it occurs,
which is what makes the trajectory path-dependent.

Ellipses are 95% confidence intervals on both axes. The phase-III onset is located where
smoothed weight–kernel similarity peaks and begins to reverse.

In [ ]:
BIO = 'Masked Eucl.'   # the bioRNN class; the only one trained with a kernel


def hysteresis(arrays, wk_label, filename, every=4):
    """Draw the trajectory of fMRI prediction against weight–kernel similarity."""
    wk_mean, wk_ci = traj.mean_ci(arrays['wk'])
    z_mean, z_ci = traj.mean_ci(arrays['z'])
    acc_mean, _ = traj.mean_ci(arrays['accuracy'])
    phases = traj.phase_onsets(wk_mean, acc_mean)

    # Accuracy spans a wide range and rises abruptly; a log scale keeps the early,
    # near-zero part of the trajectory legible.
    norm = plt.Normalize(np.log1p(0), np.log1p(acc_mean.max() * 100))
    colors = cm.plasma(norm(np.log1p(acc_mean * 100)))

    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    for i in range(len(wk_mean) - 1):
        ax.plot(wk_mean[i:i+2], z_mean[i:i+2], color=colors[i], lw=2, zorder=2)
    for i in range(0, len(wk_mean), every):
        ax.add_patch(Ellipse((wk_mean[i], z_mean[i]),
                             width=2 * wk_ci[i], height=2 * z_ci[i],
                             color=colors[i], alpha=0.15, lw=0, zorder=1))

    ax.axhspan(-1.96, 1.96, color='0.9', zorder=0)
    for idx, name in ((phases['phase_iii'], 'phase III'),
                      (phases['accuracy_onset'], 'accuracy rises')):
        ax.plot(wk_mean[idx], z_mean[idx], 'o', ms=7, mfc='none', mec='k', mew=1.4, zorder=3)
        ax.annotate(name, (wk_mean[idx], z_mean[idx]), textcoords='offset points',
                    xytext=(6, -12), fontsize=9)

    fig.colorbar(cm.ScalarMappable(norm=norm, cmap='plasma'), ax=ax,
                 label='Task accuracy (%, log scale)')
    ax.set_xlabel(f'Weight–kernel similarity ({wk_label})')
    ax.set_ylabel('fMRI variance explained\n($z$ vs random subspaces)')
    sns.despine(fig=fig, right=True, top=True)
    save(fig, filename)
    plt.show()
    return phases


phases = hysteresis(series[BIO], 'cosine', 'fig3a_hysteresis_cosine.svg')
epochs = series[BIO]['epochs']
print(f"phase III begins at epoch {epochs[phases['phase_iii']]:,}; "
      f"accuracy rises from epoch {epochs[phases['accuracy_onset']]:,}")

## Fig. S5 — the same trajectory under a rank-based metric

Repeating the loop with Spearman correlation in place of cosine confirms the shape is a
property of the trajectory rather than of the similarity measure.

In [ ]:
bio_result = results[labels.index(BIO)]
spearman = traj.epoch_arrays(bio_result, fields=('accuracy',),
                             modality=MODALITY, wk_metric='spearman')
spearman['z'] = traj.variance_explained_z(spearman, null, MODALITY)

_ = hysteresis(spearman, 'Spearman', 'figS5_hysteresis_spearman.svg')

## Fig. 3b — task-evoked versus intrinsic dynamics

The trained bioRNNs are driven two ways — by the task, and by unstructured noise — and
the fMRI variance they explain is split into what each regime uniquely accounts for and
what both share.

Each component is standardized against the null distribution of **that same component**,
drawn from random subspace *pairs*. This matters: a random pair barely overlaps, so the
shared component has a null near zero, while a lone random subspace explains ~5% of the
variance by itself. Scoring the shared component against the plain-VE null would
understate it severely.

In [ ]:
paired_null = nu.random_subspace_null(
    {'task': fmri_task, 'rest': fmri_rest},
    n_pc=N_PC, n_draws=N_NULL_DRAWS, paired=True, seed=0)

bio_model = dyn.load_model_table(MODEL_PARAMS, rows=(2,)).iloc[0]
analysis_epoch = int(trajectory['args']['max_epoch'])

components = ('unique_a', 'unique_b', 'shared')
decomposition = {c: [] for c in components}
accuracy = []
for run in tqdm(range(int(bio_model.n_runs)), desc='bioRNN runs', leave=False):
    evaluated = dyn.evaluate_run(bio_model, run, analysis_epoch, paths.model_dir,
                                 rest_nsteps=rest_nsteps, n_pc=N_PC)
    parts = dyn.variance_decomposition(evaluated, fmri_task)
    accuracy.append(evaluated['accuracy'])
    for c in components:
        decomposition[c].append(parts[c])

learned = dyn.learned_runs(accuracy)
z_components = {c: nu.ve_to_z(np.asarray(decomposition[c])[learned],
                              paired_null, MODALITY, component=c)
                for c in components}

for c in components:
    draws = paired_null[MODALITY][c]
    print(f'{c:<10} null {draws.mean():+.5f} ± {draws.std(ddof=1):.5f}   '
          f'z = {z_components[c].mean():.2f} ± {z_components[c].std(ddof=1):.2f}')

In [ ]:
names = {'unique_a': 'Unique to\ntask input',
         'unique_b': 'Unique to\nnoise input',
         'shared': 'Shared'}

fig, ax = plt.subplots(figsize=(4.6, 4))
rng = np.random.default_rng(0)

ax.axhspan(-1.96, 1.96, color='0.85', zorder=0)
ax.axhline(0, color='0.5', lw=1, zorder=1)
for i, c in enumerate(components):
    values = z_components[c]
    body = ax.violinplot(values, positions=[i], showextrema=False)
    for b in body['bodies']:
        b.set_facecolor(COLORS[i])
        b.set_alpha(0.4)
    ax.scatter(i + rng.uniform(-0.08, 0.08, values.size), values,
               s=10, color=COLORS[i], alpha=0.6, linewidths=0)
    ax.hlines(np.median(values), i - 0.22, i + 0.22, color='k', lw=2, zorder=3)

ax.set_xticks(range(len(components)))
ax.set_xticklabels([names[c] for c in components])
ax.set_ylabel('fMRI variance explained\n($z$ vs matched null)')
sns.despine(fig=fig, right=True, top=True)
save(fig, 'fig3b_variance_decomposition.svg')
plt.show()

## Reported values

In [ ]:
print('fMRI prediction across training (z vs random subspaces)')
print(f'{"class":<16}{"start":>10}{"peak":>10}{"end":>10}')
print('-' * 46)
for label in labels:
    mean, _ = traj.mean_ci(series[label]['z'])
    print(f'{label:<16}{mean[0]:>10.2f}{np.nanmax(mean):>10.2f}{mean[-1]:>10.2f}')

wk_mean, _ = traj.mean_ci(series[BIO]['wk'])
print(f'\nbioRNN weight–kernel similarity (cosine): '
      f'{wk_mean[0]:.3f} → peak {wk_mean[phases["phase_iii"]]:.3f} → {wk_mean[-1]:.3f}')
print(f'  phase III from epoch {epochs[phases["phase_iii"]]:,}; '
      f'accuracy rises from epoch {epochs[phases["accuracy_onset"]]:,}')

print('\nVariance decomposition (bioRNN, task fMRI)')
for c in components:
    z = z_components[c]
    print(f'  {names[c].replace(chr(10), " "):<22} z = {z.mean():5.2f} ± {z.std(ddof=1):.2f}')
print(f'\nEvaluated at epoch {analysis_epoch:,}; {N_FMRI_SUBJ} subjects; '
      f'null = {N_NULL_DRAWS} random {N_PC}-D subspaces (seed 0).')